# 06. Database Integration

## AI-Powered Customer Intelligence and Voice of Customer Analytics Platform

### Project Overview

This notebook focuses on integrating the customer intelligence analysis results with a PostgreSQL database.

The database integration process will help store, manage and retrieve customer review data, sentiment analysis results, feedback themes and business recommendations efficiently.

### Objectives

1. Establish a connection between Python and PostgreSQL.
2. Create the required database tables.
3. Import processed customer analysis datasets.
4. Store business recommendations in the database.
5. Execute SQL queries for customer data analysis.
6. Generate useful business insights using SQL.
7. Prepare database views for dashboard integration.

### Database Details

- Database Management System: PostgreSQL
- Database Name: voc_analytics_db
- Python Environment: Jupyter Notebook
- Main Data Source: Processed Customer Review Datasets

## 2. Import Required Libraries

We will import the Python libraries required for database integration and data handling.

- pandas: To load and manage CSV datasets.
- pathlib: To manage project directories and file paths.
- importlib: To check the availability of required packages.

In [1]:
# Import required libraries
import pandas as pd
from pathlib import Path
import importlib.util

print("Libraries imported successfully!")
print("Pandas Version:", pd.__version__)

Libraries imported successfully!
Pandas Version: 3.0.1


## 3. Define Project Directory

In this step, we will identify the main project directory and locate the processed datasets.

The processed data directory contains the final customer analysis and business recommendation files generated in the previous notebooks.

In [2]:
# Identify the main project directory
current_path = Path.cwd()

PROJECT_ROOT = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / "data" / "processed").exists()
        and (path / "notebooks").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project directory not found. Please check the notebook location."
    )

# Define processed data directory
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project Directory:", PROJECT_ROOT)
print("Processed Data Directory:", PROCESSED_DIR)

# Display available datasets
print("\nAvailable Processed Datasets:")

for file in sorted(PROCESSED_DIR.glob("*.csv")):
    print(file.name)

Project Directory: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence
Processed Data Directory: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\data\processed

Available Processed Datasets:
business_recommendations.csv
cleaned_customer_reviews.csv
final_customer_analysis.csv
negative_sentiment_by_theme.csv
sentiment_analysis_results.csv
theme_analysis_results.csv
theme_distribution.csv


## 4. Load Final Project Datasets

In this step, we will load the two main output datasets generated during the project analysis.

1. final_customer_analysis.csv
2. business_recommendations.csv

These datasets will be used to populate the PostgreSQL database.

In [3]:
# Load final customer analysis dataset
final_customer_df = pd.read_csv(
    PROCESSED_DIR / "final_customer_analysis.csv"
)

# Load business recommendations dataset
business_recommendations_df = pd.read_csv(
    PROCESSED_DIR / "business_recommendations.csv"
)

# Display dataset information
print("CUSTOMER ANALYSIS DATASET")
print("Shape:", final_customer_df.shape)
print("Total Records:", len(final_customer_df))
print("Total Columns:", len(final_customer_df.columns))

print("\nBUSINESS RECOMMENDATIONS DATASET")
print("Shape:", business_recommendations_df.shape)
print("Total Records:", len(business_recommendations_df))
print("Total Columns:", len(business_recommendations_df.columns))

print("\nDatasets loaded successfully!")

CUSTOMER ANALYSIS DATASET
Shape: (28332, 30)
Total Records: 28332
Total Columns: 30

BUSINESS RECOMMENDATIONS DATASET
Shape: (5, 3)
Total Records: 5
Total Columns: 3

Datasets loaded successfully!


## 5. Check PostgreSQL Connection Libraries

To connect Python with PostgreSQL, we need the following packages:

1. SQLAlchemy – To manage database connections and transfer datasets.
2. psycopg2 – To communicate with the PostgreSQL database.

In this step, we will check whether these packages are already installed in the Jupyter environment.

In [4]:
# Check required database packages
required_packages = ["sqlalchemy", "psycopg2"]

for package in required_packages:
    is_installed = importlib.util.find_spec(package) is not None

    if is_installed:
        print(f"{package}: Installed")
    else:
        print(f"{package}: Not Installed")

sqlalchemy: Installed
psycopg2: Installed


## 6. Configure PostgreSQL Database Connection

In this step, we will establish a connection between Python and the PostgreSQL database using SQLAlchemy.

Database Configuration:

- Database System: PostgreSQL
- Host: localhost
- Port: 5433
- Database Name: voc_analytics_db
- Username: postgres

The database password will be entered securely.

In [5]:
# Import database connection libraries
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

# PostgreSQL Database Configuration
DB_HOST = "localhost"
DB_PORT = 5433
DB_NAME = "voc_analytics_db"
DB_USER = "postgres"

# Securely enter PostgreSQL password
DB_PASSWORD = getpass("Enter PostgreSQL Password: ")

# Create PostgreSQL connection URL
DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

# Create SQLAlchemy engine
engine = create_engine(DATABASE_URL)

# Verify PostgreSQL connection
try:
    with engine.connect() as connection:

        result = connection.execute(
            text("SELECT current_database(), current_user, inet_server_port();")
        )

        database_name, username, port = result.fetchone()

        print("PostgreSQL Connection Successful!")
        print("----------------------------------")
        print("Database Name:", database_name)
        print("Username:", username)
        print("Port:", port)

except Exception as error:
    print("PostgreSQL Connection Failed!")
    print("Error:", error)

Enter PostgreSQL Password:  ········


PostgreSQL Connection Successful!
----------------------------------
Database Name: voc_analytics_db
Username: postgres
Port: 5433


## 7. Create Database Tables

In this step, we will create two tables in the PostgreSQL database.

1. customer_reviews – Stores customer review details, product information, sentiment scores and feedback themes.
2. business_recommendations – Stores business findings and recommended actions.

These tables will provide a structured foundation for storing and querying the project data.

In [6]:
# SQL query to create customer reviews table
create_customer_table = """
CREATE TABLE IF NOT EXISTS customer_reviews (
    review_id BIGSERIAL PRIMARY KEY,
    product_id TEXT,
    date_added TEXT,
    date_updated TEXT,
    product_name TEXT,
    asins TEXT,
    brand TEXT,
    categories TEXT,
    primary_categories TEXT,
    image_urls TEXT,
    keys TEXT,
    manufacturer TEXT,
    manufacturer_number TEXT,
    review_date_original TEXT,
    review_date_seen TEXT,
    do_recommend TEXT,
    num_helpful DOUBLE PRECISION,
    rating INTEGER,
    review_source_urls TEXT,
    review_text TEXT,
    review_title TEXT,
    review_username TEXT,
    source_urls TEXT,
    review_text_clean TEXT,
    review_date_clean TEXT,
    vader_negative DOUBLE PRECISION,
    vader_neutral DOUBLE PRECISION,
    vader_positive DOUBLE PRECISION,
    sentiment_score DOUBLE PRECISION,
    sentiment_label TEXT,
    matched_themes TEXT
);
"""

# SQL query to create business recommendations table
create_recommendations_table = """
CREATE TABLE IF NOT EXISTS business_recommendations (
    recommendation_id BIGSERIAL PRIMARY KEY,
    area TEXT,
    finding TEXT,
    recommendation TEXT
);
"""

# Execute table creation queries
with engine.begin() as connection:
    connection.execute(text(create_customer_table))
    connection.execute(text(create_recommendations_table))

print("Database tables created successfully!")
print("1. customer_reviews")
print("2. business_recommendations")

Database tables created successfully!
1. customer_reviews
2. business_recommendations


## 8. Prepare Customer Data for PostgreSQL

In this step, we will prepare the final customer analysis dataset for PostgreSQL integration.

The original dataset contains 30 columns. We will rename these columns into database-friendly names that match the customer_reviews table.

The original CSV files will remain unchanged.

In [10]:
# Create a copy of the customer analysis dataset
customer_db_df = final_customer_df.copy()

# Rename columns to match PostgreSQL table structure
column_mapping = {
    "id": "product_id",
    "dateAdded": "date_added",
    "dateUpdated": "date_updated",
    "name": "product_name",
    "asins": "asins",
    "brand": "brand",
    "categories": "categories",
    "primaryCategories": "primary_categories",
    "imageURLs": "image_urls",
    "keys": "keys",
    "manufacturer": "manufacturer",
    "manufacturerNumber": "manufacturer_number",
    "reviews.date": "review_date_original",
    "reviews.dateSeen": "review_date_seen",
    "reviews.doRecommend": "do_recommend",
    "reviews.numHelpful": "num_helpful",
    "reviews.rating": "rating",
    "reviews.sourceURLs": "review_source_urls",
    "reviews.text": "review_text",
    "reviews.title": "review_title",
    "reviews.username": "review_username",
    "sourceURLs": "source_urls",
    "review_text_clean": "review_text_clean",
    "review_date": "review_date_clean",
    "vader_negative": "vader_negative",
    "vader_neutral": "vader_neutral",
    "vader_positive": "vader_positive",
    "sentiment_score": "sentiment_score",
    "sentiment_label": "sentiment_label",
    "matched_themes": "matched_themes"
}

customer_db_df = customer_db_df.rename(columns=column_mapping)

# Convert recommendation values into database-compatible text
customer_db_df["do_recommend"] = customer_db_df["do_recommend"].map(
    lambda value: None if pd.isna(value) else str(value)
)

# Verify prepared dataset
print("Customer Dataset Prepared Successfully!")
print("----------------------------------------")
print("Total Records:", len(customer_db_df))
print("Total Columns:", len(customer_db_df.columns))

print("\nDatabase Column Names:")
print(customer_db_df.columns.tolist())

print("\nSample Records:")
display(customer_db_df.head(3))

Customer Dataset Prepared Successfully!
----------------------------------------
Total Records: 28332
Total Columns: 30

Database Column Names:
['product_id', 'date_added', 'date_updated', 'product_name', 'asins', 'brand', 'categories', 'primary_categories', 'image_urls', 'keys', 'manufacturer', 'manufacturer_number', 'review_date_original', 'review_date_seen', 'do_recommend', 'num_helpful', 'rating', 'review_source_urls', 'review_text', 'review_title', 'review_username', 'source_urls', 'review_text_clean', 'review_date_clean', 'vader_negative', 'vader_neutral', 'vader_positive', 'sentiment_score', 'sentiment_label', 'matched_themes']

Sample Records:


,product_id,date_added,date_updated,product_name,asins,brand,categories,primary_categories,image_urls,keys,...,review_username,source_urls,review_text_clean,review_date_clean,vader_negative,vader_neutral,vader_positive,sentiment_score,sentiment_label,matched_themes
0,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,Byger yang,"https://www.barcodable.com/upc/841710106442,ht...",i order 3 of them and one of the item is bad q...,2017-03-02 00:00:00+00:00,0.186,0.814,0.000,-0.6908,Negative,Product Quality
1,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,ByMG,"https://www.barcodable.com/upc/841710106442,ht...",bulk is always the less expensive way to go fo...,2016-08-31 00:00:00+00:00,0.000,0.828,0.172,0.3612,Positive,Price and Value
2,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,BySharon Lambert,"https://www.barcodable.com/upc/841710106442,ht...",well they are not duracell but for the price i...,2016-12-06 00:00:00+00:00,0.000,0.577,0.423,0.7650,Positive,Price and Value


## 9. Prepare Business Recommendations Data

In this step, we will prepare the business recommendations dataset for PostgreSQL integration.

The dataset contains important business findings and recommended actions identified during the customer feedback analysis.

We will rename the columns to match the business_recommendations table structure.

In [11]:
# Create a copy of the business recommendations dataset
recommendations_db_df = business_recommendations_df.copy()

# Rename columns to match PostgreSQL table structure
recommendations_db_df = recommendations_db_df.rename(
    columns={
        "Area": "area",
        "Finding": "finding",
        "Recommendation": "recommendation"
    }
)

# Verify the prepared dataset
print("Business Recommendations Prepared Successfully!")
print("-----------------------------------------------")
print("Total Records:", len(recommendations_db_df))
print("Total Columns:", len(recommendations_db_df.columns))

print("\nDatabase Column Names:")
print(recommendations_db_df.columns.tolist())

print("\nPrepared Business Recommendations:")
display(recommendations_db_df)

Business Recommendations Prepared Successfully!
-----------------------------------------------
Total Records: 5
Total Columns: 3

Database Column Names:
['area', 'finding', 'recommendation']

Prepared Business Recommendations:


,area,finding,recommendation
0,Customer Service,Highest negative sentiment percentage among id...,Improve customer support response time and com...
1,Health & Beauty,Highest negative feedback percentage among pro...,Investigate product-specific complaints and im...
2,Price and Value,Most frequently discussed identified feedback ...,"Review product pricing, affordability, and val..."
3,Delivery Experience,Negative feedback is present in delivery-relat...,Monitor shipping delays and improve delivery c...
4,Unidentified Feedback,"1,295 negative reviews belong to the Other theme.",Expand keyword coverage and investigate unclas...


## 10. Validate Prepared Datasets

In this step, we will validate both prepared datasets before importing them into PostgreSQL.

We will check the column names, column order, and record counts to ensure that both datasets match the database table structure.

In [12]:
# Expected columns for customer reviews table
customer_expected_columns = [
    "product_id",
    "date_added",
    "date_updated",
    "product_name",
    "asins",
    "brand",
    "categories",
    "primary_categories",
    "image_urls",
    "keys",
    "manufacturer",
    "manufacturer_number",
    "review_date_original",
    "review_date_seen",
    "do_recommend",
    "num_helpful",
    "rating",
    "review_source_urls",
    "review_text",
    "review_title",
    "review_username",
    "source_urls",
    "review_text_clean",
    "review_date_clean",
    "vader_negative",
    "vader_neutral",
    "vader_positive",
    "sentiment_score",
    "sentiment_label",
    "matched_themes"
]

# Expected columns for business recommendations table
recommendations_expected_columns = [
    "area",
    "finding",
    "recommendation"
]

# Validate customer reviews dataset
assert list(customer_db_df.columns) == customer_expected_columns, \
    "Customer reviews column mismatch!"

# Validate business recommendations dataset
assert list(recommendations_db_df.columns) == recommendations_expected_columns, \
    "Business recommendations column mismatch!"

# Display validation results
print("Dataset Validation Successful!")
print("--------------------------------")
print("Customer Reviews Records:", len(customer_db_df))
print("Customer Reviews Columns:", len(customer_db_df.columns))

print("\nBusiness Recommendations Records:", len(recommendations_db_df))
print("Business Recommendations Columns:", len(recommendations_db_df.columns))

print("\nBoth datasets match the PostgreSQL table structure.")

Dataset Validation Successful!
--------------------------------
Customer Reviews Records: 28332
Customer Reviews Columns: 30

Business Recommendations Records: 5
Business Recommendations Columns: 3

Both datasets match the PostgreSQL table structure.


## 11. Import Datasets into PostgreSQL

In this step, we will import the prepared customer reviews and business recommendations datasets into their respective PostgreSQL tables.

We will use Pandas to_sql() method to transfer the data into the database.

Existing table records will be cleared before importing to prevent duplicate records when this step is executed again.

In [13]:
# Import both datasets into PostgreSQL

with engine.begin() as connection:

    # Clear existing records to prevent duplicates
    connection.execute(
        text("TRUNCATE TABLE customer_reviews RESTART IDENTITY")
    )

    connection.execute(
        text("TRUNCATE TABLE business_recommendations RESTART IDENTITY")
    )

    # Import customer reviews dataset
    customer_db_df.to_sql(
        name="customer_reviews",
        con=connection,
        if_exists="append",
        index=False,
        chunksize=500,
        method="multi"
    )

    # Import business recommendations dataset
    recommendations_db_df.to_sql(
        name="business_recommendations",
        con=connection,
        if_exists="append",
        index=False,
        chunksize=500,
        method="multi"
    )

print("Data Imported into PostgreSQL Successfully!")
print("--------------------------------------------")
print("Customer Reviews Imported:", len(customer_db_df))
print("Business Recommendations Imported:", len(recommendations_db_df))
print("Database: voc_analytics_db")

Data Imported into PostgreSQL Successfully!
--------------------------------------------
Customer Reviews Imported: 28332
Business Recommendations Imported: 5
Database: voc_analytics_db


## 12. Verify Imported Database Records

In this step, we will verify whether both datasets have been successfully stored in PostgreSQL.

We will check the total number of records in each table and compare them with the original dataset counts.

In [14]:
# Verify records stored in PostgreSQL

with engine.connect() as connection:

    customer_count = connection.execute(
        text("SELECT COUNT(*) FROM customer_reviews")
    ).scalar_one()

    recommendations_count = connection.execute(
        text("SELECT COUNT(*) FROM business_recommendations")
    ).scalar_one()

print("PostgreSQL Data Verification")
print("--------------------------------")
print("Customer Reviews in Database:", customer_count)
print("Expected Customer Reviews:", len(customer_db_df))

print("\nBusiness Recommendations in Database:", recommendations_count)
print("Expected Business Recommendations:", len(recommendations_db_df))

# Check whether database counts match original datasets
if (
    customer_count == len(customer_db_df)
    and recommendations_count == len(recommendations_db_df)
):
    print("\nDatabase Verification Successful!")
    print("Both tables contain the expected number of records.")
else:
    print("\nDatabase Verification Failed!")
    print("Please check the imported data.")

PostgreSQL Data Verification
--------------------------------
Customer Reviews in Database: 28332
Expected Customer Reviews: 28332

Business Recommendations in Database: 5
Expected Business Recommendations: 5

Database Verification Successful!
Both tables contain the expected number of records.


## 13. Retrieve Sample Records from PostgreSQL

In this step, we will retrieve sample customer reviews and business recommendations directly from PostgreSQL.

This will help us verify that the imported data can be accessed and displayed correctly using SQL queries.

In [15]:
# Retrieve sample customer reviews
with engine.connect() as connection:

    customer_sample = pd.read_sql(
        text("""
            SELECT
                review_id,
                product_name,
                rating,
                sentiment_label,
                matched_themes
            FROM customer_reviews
            ORDER BY review_id
            LIMIT 5;
        """),
        connection
    )

    # Retrieve business recommendations
    recommendations_sample = pd.read_sql(
        text("""
            SELECT
                area,
                finding,
                recommendation
            FROM business_recommendations;
        """),
        connection
    )

print("Sample Customer Reviews from PostgreSQL")
print("----------------------------------------")
display(customer_sample)

print("\nBusiness Recommendations from PostgreSQL")
print("-----------------------------------------")
display(recommendations_sample)


Sample Customer Reviews from PostgreSQL
----------------------------------------


,review_id,product_name,rating,sentiment_label,matched_themes
0,1,AmazonBasics AAA Performance Alkaline Batterie...,3,Negative,Product Quality
1,2,AmazonBasics AAA Performance Alkaline Batterie...,4,Positive,Price and Value
2,3,AmazonBasics AAA Performance Alkaline Batterie...,5,Positive,Price and Value
3,4,AmazonBasics AAA Performance Alkaline Batterie...,5,Positive,Price and Value
4,5,AmazonBasics AAA Performance Alkaline Batterie...,5,Positive,Price and Value



Business Recommendations from PostgreSQL
-----------------------------------------


,area,finding,recommendation
0,Customer Service,Highest negative sentiment percentage among id...,Improve customer support response time and com...
1,Health & Beauty,Highest negative feedback percentage among pro...,Investigate product-specific complaints and im...
2,Price and Value,Most frequently discussed identified feedback ...,"Review product pricing, affordability, and val..."
3,Delivery Experience,Negative feedback is present in delivery-relat...,Monitor shipping delays and improve delivery c...
4,Unidentified Feedback,"1,295 negative reviews belong to the Other theme.",Expand keyword coverage and investigate unclas...


## 14. Analyze Sentiment Distribution Using SQL

In this step, we will perform sentiment analysis directly on the PostgreSQL database using SQL queries.

We will calculate the total number and percentage of Positive, Neutral, and Negative customer reviews.

In [16]:
# Analyze sentiment distribution using PostgreSQL

with engine.connect() as connection:

    sentiment_distribution = pd.read_sql(
        text("""
            SELECT
                sentiment_label,
                COUNT(*) AS total_reviews,
                ROUND(
                    COUNT(*) * 100.0 /
                    (SELECT COUNT(*) FROM customer_reviews),
                    2
                ) AS percentage
            FROM customer_reviews
            GROUP BY sentiment_label
            ORDER BY total_reviews DESC;
        """),
        connection
    )

print("Sentiment Distribution from PostgreSQL")
print("--------------------------------------")
display(sentiment_distribution)

Sentiment Distribution from PostgreSQL
--------------------------------------


,sentiment_label,total_reviews,percentage
0,Positive,23724,83.74
1,Neutral,2498,8.82
2,Negative,2110,7.45


## 15. Analyze Product Ratings Using SQL

In this step, we will analyze customer product ratings directly from PostgreSQL.

We will calculate the average product rating and examine the distribution of reviews across different rating values.

In [17]:
# Analyze average rating and rating distribution

with engine.connect() as connection:

    average_rating = pd.read_sql(
        text("""
            SELECT
                ROUND(AVG(rating)::NUMERIC, 2) AS average_rating
            FROM customer_reviews;
        """),
        connection
    )

    rating_distribution = pd.read_sql(
        text("""
            SELECT
                rating,
                COUNT(*) AS total_reviews,
                ROUND(
                    COUNT(*) * 100.0 /
                    (SELECT COUNT(*) FROM customer_reviews),
                    2
                ) AS percentage
            FROM customer_reviews
            GROUP BY rating
            ORDER BY rating;
        """),
        connection
    )

print("Average Product Rating")
print("----------------------")
display(average_rating)

print("\nRating-wise Distribution")
print("------------------------")
display(rating_distribution)

Average Product Rating
----------------------


,average_rating
0,4.51



Rating-wise Distribution
------------------------


,rating,total_reviews,percentage
0,1,965,3.41
1,2,616,2.17
2,3,1206,4.26
3,4,5648,19.94
4,5,19897,70.23


## 16. Analyze Customer Feedback Themes Using SQL

In this step, we will analyze customer feedback themes stored in PostgreSQL.

We will identify the most frequently discussed themes and calculate the number of reviews associated with each theme.

This analysis will help us understand the major areas of customer feedback.

In [18]:
# Analyze customer feedback themes using PostgreSQL

with engine.connect() as connection:

    theme_distribution = pd.read_sql(
        text("""
            SELECT
                matched_themes AS theme,
                COUNT(*) AS total_reviews
            FROM customer_reviews
            WHERE matched_themes IS NOT NULL
            AND TRIM(matched_themes) != ''
            GROUP BY matched_themes
            ORDER BY total_reviews DESC;
        """),
        connection
    )

print("Customer Feedback Theme Distribution")
print("-------------------------------------")
display(theme_distribution)

Customer Feedback Theme Distribution
-------------------------------------


,theme,total_reviews
0,Other,16852
1,Price and Value,5649
2,Product Performance,1825
3,Product Quality,839
4,Price and Value; Product Performance,716
5,Product Quality; Price and Value,603
6,Packaging,325
7,Customer Service,232
8,Price and Value; Packaging,229
9,Delivery Experience,214


## 17. Analyze Negative Feedback by Theme

In this step, we will analyze negative customer reviews stored in PostgreSQL.

We will calculate the number of negative reviews associated with each theme combination.

This analysis will help identify areas where customers are experiencing problems.

In [19]:
# Analyze negative feedback by theme

with engine.connect() as connection:

    negative_theme_analysis = pd.read_sql(
        text("""
            SELECT
                matched_themes AS theme,
                COUNT(*) AS negative_reviews
            FROM customer_reviews
            WHERE sentiment_label = 'Negative'
            GROUP BY matched_themes
            ORDER BY negative_reviews DESC;
        """),
        connection
    )

print("Negative Customer Feedback by Theme")
print("------------------------------------")
display(negative_theme_analysis.head(15))

Negative Customer Feedback by Theme
------------------------------------


,theme,negative_reviews
0,Other,1295
1,Price and Value,319
2,Product Performance,145
3,Product Quality,95
4,Price and Value; Product Performance,50
5,Packaging,45
6,Customer Service,28
7,Delivery Experience,28
8,Product Quality; Price and Value,18
9,Price and Value; Packaging,18


## 18. Save Database Table Creation SQL Script

In this step, we will save the SQL commands used to create the PostgreSQL database tables into a separate SQL script.

This script will be stored in the project's sql folder for documentation, maintenance, and future database setup.

In [20]:
# Create SQL directory if it does not exist
SQL_DIR = PROJECT_ROOT / "sql"
SQL_DIR.mkdir(parents=True, exist_ok=True)

# Define SQL table creation commands
create_tables_sql = """
-- Customer Reviews Table

CREATE TABLE IF NOT EXISTS customer_reviews (
    review_id BIGSERIAL PRIMARY KEY,
    product_id TEXT,
    date_added TEXT,
    date_updated TEXT,
    product_name TEXT,
    asins TEXT,
    brand TEXT,
    categories TEXT,
    primary_categories TEXT,
    image_urls TEXT,
    keys TEXT,
    manufacturer TEXT,
    manufacturer_number TEXT,
    review_date_original TEXT,
    review_date_seen TEXT,
    do_recommend TEXT,
    num_helpful DOUBLE PRECISION,
    rating INTEGER,
    review_source_urls TEXT,
    review_text TEXT,
    review_title TEXT,
    review_username TEXT,
    source_urls TEXT,
    review_text_clean TEXT,
    review_date_clean TEXT,
    vader_negative DOUBLE PRECISION,
    vader_neutral DOUBLE PRECISION,
    vader_positive DOUBLE PRECISION,
    sentiment_score DOUBLE PRECISION,
    sentiment_label TEXT,
    matched_themes TEXT
);

-- Business Recommendations Table

CREATE TABLE IF NOT EXISTS business_recommendations (
    recommendation_id BIGSERIAL PRIMARY KEY,
    area TEXT,
    finding TEXT,
    recommendation TEXT
);
"""

# Save SQL script
sql_file = SQL_DIR / "01_create_tables.sql"
sql_file.write_text(create_tables_sql.strip() + "\n", encoding="utf-8")

print("SQL Script Saved Successfully!")
print("--------------------------------")
print("File Name:", sql_file.name)
print("File Location:", sql_file)

SQL Script Saved Successfully!
--------------------------------
File Name: 01_create_tables.sql
File Location: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\sql\01_create_tables.sql


## 19. Save SQL Data Analysis Queries

In this step, we will save the SQL queries used for sentiment analysis, product rating analysis, customer feedback theme distribution, and negative feedback analysis.

These queries will be stored in a separate SQL script for future use and reporting.

In [21]:
# SQL queries for project analysis

data_analysis_sql = """
-- 1. Sentiment Distribution

SELECT
    sentiment_label,
    COUNT(*) AS total_reviews,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM customer_reviews),
        2
    ) AS percentage
FROM customer_reviews
GROUP BY sentiment_label
ORDER BY total_reviews DESC;


-- 2. Average Product Rating

SELECT
    ROUND(AVG(rating)::NUMERIC, 2) AS average_rating
FROM customer_reviews;


-- 3. Rating-wise Distribution

SELECT
    rating,
    COUNT(*) AS total_reviews,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM customer_reviews),
        2
    ) AS percentage
FROM customer_reviews
GROUP BY rating
ORDER BY rating;


-- 4. Customer Feedback Theme Distribution

SELECT
    matched_themes AS theme,
    COUNT(*) AS total_reviews
FROM customer_reviews
WHERE matched_themes IS NOT NULL
AND TRIM(matched_themes) != ''
GROUP BY matched_themes
ORDER BY total_reviews DESC;


-- 5. Negative Feedback by Theme

SELECT
    matched_themes AS theme,
    COUNT(*) AS negative_reviews
FROM customer_reviews
WHERE sentiment_label = 'Negative'
GROUP BY matched_themes
ORDER BY negative_reviews DESC;


-- 6. Total Customer Reviews

SELECT COUNT(*) AS total_reviews
FROM customer_reviews;


-- 7. Business Recommendations

SELECT *
FROM business_recommendations;
"""

# Save SQL analysis script
analysis_sql_file = SQL_DIR / "02_data_analysis_queries.sql"

analysis_sql_file.write_text(
    data_analysis_sql.strip() + "\n",
    encoding="utf-8"
)

print("SQL Analysis Queries Saved Successfully!")
print("----------------------------------------")
print("File Name:", analysis_sql_file.name)
print("File Location:", analysis_sql_file)

SQL Analysis Queries Saved Successfully!
----------------------------------------
File Name: 02_data_analysis_queries.sql
File Location: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\sql\02_data_analysis_queries.sql


## 20. Save Business Insights SQL Script

In this step, we will save SQL queries for extracting important business insights from customer reviews.

These queries will focus on customer satisfaction, negative feedback, product categories, and business recommendations.

In [22]:
# SQL queries for business insights

business_insights_sql = """
-- 1. Overall Customer Sentiment Summary

SELECT
    sentiment_label,
    COUNT(*) AS total_reviews
FROM customer_reviews
GROUP BY sentiment_label
ORDER BY total_reviews DESC;


-- 2. Customer Satisfaction by Product Rating

SELECT
    rating,
    COUNT(*) AS total_reviews
FROM customer_reviews
GROUP BY rating
ORDER BY rating DESC;


-- 3. Negative Reviews by Product Category

SELECT
    primary_categories,
    COUNT(*) AS negative_reviews
FROM customer_reviews
WHERE sentiment_label = 'Negative'
GROUP BY primary_categories
ORDER BY negative_reviews DESC
LIMIT 10;


-- 4. Most Frequently Discussed Themes

SELECT
    matched_themes,
    COUNT(*) AS total_reviews
FROM customer_reviews
WHERE matched_themes IS NOT NULL
GROUP BY matched_themes
ORDER BY total_reviews DESC
LIMIT 10;


-- 5. Business Recommendations

SELECT
    area,
    finding,
    recommendation
FROM business_recommendations
ORDER BY area;
"""

# Save business insights SQL script
business_insights_file = SQL_DIR / "03_business_insights.sql"

business_insights_file.write_text(
    business_insights_sql.strip() + "\n",
    encoding="utf-8"
)

print("Business Insights SQL Script Saved Successfully!")
print("------------------------------------------------")
print("File Name:", business_insights_file.name)
print("File Location:", business_insights_file)

Business Insights SQL Script Saved Successfully!
------------------------------------------------
File Name: 03_business_insights.sql
File Location: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\sql\03_business_insights.sql


## 21. Create SQL Database Views

In this step, we will create reusable PostgreSQL views for customer sentiment distribution, product rating distribution, and negative customer feedback.

These views will simplify future data analysis and dashboard integration.

In [23]:
# SQL queries for creating database views

views_sql = """
-- 1. Customer Sentiment Summary View

CREATE OR REPLACE VIEW customer_sentiment_summary AS
SELECT
    sentiment_label,
    COUNT(*) AS total_reviews,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM customer_reviews),
        2
    ) AS percentage
FROM customer_reviews
GROUP BY sentiment_label;


-- 2. Product Rating Summary View

CREATE OR REPLACE VIEW product_rating_summary AS
SELECT
    rating,
    COUNT(*) AS total_reviews
FROM customer_reviews
GROUP BY rating;


-- 3. Negative Feedback Summary View

CREATE OR REPLACE VIEW negative_feedback_summary AS
SELECT
    matched_themes AS theme,
    COUNT(*) AS negative_reviews
FROM customer_reviews
WHERE sentiment_label = 'Negative'
GROUP BY matched_themes;


-- 4. Business Recommendations View

CREATE OR REPLACE VIEW business_recommendations_summary AS
SELECT
    area,
    finding,
    recommendation
FROM business_recommendations;
"""

# Save SQL views script
views_file = SQL_DIR / "04_views.sql"

views_file.write_text(
    views_sql.strip() + "\n",
    encoding="utf-8"
)

print("SQL Views Script Saved Successfully!")
print("-------------------------------------")
print("File Name:", views_file.name)
print("File Location:", views_file)

SQL Views Script Saved Successfully!
-------------------------------------
File Name: 04_views.sql
File Location: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\sql\04_views.sql


## 22. Execute SQL Views in PostgreSQL

In this step, we will execute the saved SQL views script in PostgreSQL.

This will create reusable database views for sentiment analysis, product ratings, negative feedback, and business recommendations.

In [24]:
# Read the saved SQL views script
views_sql_content = views_file.read_text(encoding="utf-8")

# Execute all view creation statements
with engine.begin() as connection:
    connection.execute(text(views_sql_content))

print("PostgreSQL Views Created Successfully!")
print("--------------------------------------")
print("1. customer_sentiment_summary")
print("2. product_rating_summary")
print("3. negative_feedback_summary")
print("4. business_recommendations_summary")

PostgreSQL Views Created Successfully!
--------------------------------------
1. customer_sentiment_summary
2. product_rating_summary
3. negative_feedback_summary
4. business_recommendations_summary


## 23. Verify PostgreSQL Views

In this step, we will verify all four PostgreSQL views by retrieving records from each view.

This will confirm that the views have been created correctly and are returning the expected results.

In [25]:
# Verify all PostgreSQL views

with engine.connect() as connection:

    sentiment_view = pd.read_sql(
        text("SELECT * FROM customer_sentiment_summary ORDER BY total_reviews DESC"),
        connection
    )

    rating_view = pd.read_sql(
        text("SELECT * FROM product_rating_summary ORDER BY rating"),
        connection
    )

    negative_view = pd.read_sql(
        text("SELECT * FROM negative_feedback_summary ORDER BY negative_reviews DESC LIMIT 10"),
        connection
    )

    recommendations_view = pd.read_sql(
        text("SELECT * FROM business_recommendations_summary"),
        connection
    )

print("1. Customer Sentiment Summary View")
print("----------------------------------")
display(sentiment_view)

print("\n2. Product Rating Summary View")
print("-------------------------------")
display(rating_view)

print("\n3. Negative Feedback Summary View")
print("---------------------------------")
display(negative_view)

print("\n4. Business Recommendations Summary View")
print("-----------------------------------------")
display(recommendations_view)

print("\nAll PostgreSQL Views Verified Successfully!")

1. Customer Sentiment Summary View
----------------------------------


,sentiment_label,total_reviews,percentage
0,Positive,23724,83.74
1,Neutral,2498,8.82
2,Negative,2110,7.45



2. Product Rating Summary View
-------------------------------


,rating,total_reviews
0,1,965
1,2,616
2,3,1206
3,4,5648
4,5,19897



3. Negative Feedback Summary View
---------------------------------


,theme,negative_reviews
0,Other,1295
1,Price and Value,319
2,Product Performance,145
3,Product Quality,95
4,Price and Value; Product Performance,50
5,Packaging,45
6,Customer Service,28
7,Delivery Experience,28
8,Price and Value; Packaging,18
9,Product Quality; Price and Value,18



4. Business Recommendations Summary View
-----------------------------------------


,area,finding,recommendation
0,Customer Service,Highest negative sentiment percentage among id...,Improve customer support response time and com...
1,Health & Beauty,Highest negative feedback percentage among pro...,Investigate product-specific complaints and im...
2,Price and Value,Most frequently discussed identified feedback ...,"Review product pricing, affordability, and val..."
3,Delivery Experience,Negative feedback is present in delivery-relat...,Monitor shipping delays and improve delivery c...
4,Unidentified Feedback,"1,295 negative reviews belong to the Other theme.",Expand keyword coverage and investigate unclas...



All PostgreSQL Views Verified Successfully!


# 24. Final Database Integration Summary

## Project: AI-Powered Customer Intelligence and Voice of Customer Analytics Platform

### Database Information
- Database: voc_analytics_db
- Database System: PostgreSQL
- Host: localhost
- Port: 5433

### Database Tables
1. customer_reviews
2. business_recommendations

### Imported Data
- Total Customer Reviews: 28,332
- Total Business Recommendations: 5

### SQL Analysis Completed
- Customer Sentiment Distribution
- Average Product Rating
- Rating-wise Distribution
- Customer Feedback Theme Distribution
- Negative Feedback Analysis
- Business Insights

### PostgreSQL Views Created
1. customer_sentiment_summary
2. product_rating_summary
3. negative_feedback_summary
4. business_recommendations_summary

### SQL Scripts Created
1. 01_create_tables.sql
2. 02_data_analysis_queries.sql
3. 03_business_insights.sql
4. 04_views.sql

### Final Status

Database connection, table creation, data import, record verification, SQL analysis, and database view creation have been completed successfully.

The PostgreSQL database is now ready for integration with the project's dashboard application.

## Streamlit 

In [27]:
%pip show streamlit

Name: streamlit
Version: 1.51.0
Summary: A faster way to build and share data apps
Home-page: https://streamlit.io
Author: Snowflake Inc
Author-email: hello@streamlit.io
License: Apache License 2.0
Location: C:\Users\anant\anaconda3\Lib\site-packages
Requires: altair, blinker, cachetools, click, gitpython, numpy, packaging, pandas, pillow, protobuf, pyarrow, requests, tenacity, toml, tornado, typing-extensions, watchdog
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## 26. Create Dashboard Application File

In this step, we will create a separate Python file for our Streamlit dashboard application.

In [28]:
# Create dashboard application file

DASHBOARD_DIR = PROJECT_ROOT / "dashboard"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

dashboard_file = DASHBOARD_DIR / "app.py"

dashboard_file.write_text(
    "# AI-Powered Customer Intelligence and Voice of Customer Analytics Platform\n",
    encoding="utf-8"
)

print("Dashboard Application File Created Successfully!")
print("-----------------------------------------------")
print("File Name:", dashboard_file.name)
print("File Location:", dashboard_file)

Dashboard Application File Created Successfully!
-----------------------------------------------
File Name: app.py
File Location: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py


## 27. Import Dashboard Libraries

In this step, we will prepare the required Python libraries for developing our interactive Streamlit dashboard.

In [29]:
# Import dashboard libraries

import streamlit as st
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

print("Dashboard Libraries Imported Successfully!")
print("--------------------------------------------")
print("Streamlit Version:", st.__version__)
print("Pandas Version:", pd.__version__)

Dashboard Libraries Imported Successfully!
--------------------------------------------
Streamlit Version: 1.51.0
Pandas Version: 3.0.1


## 28. Configure Dashboard Database Connection

In this step, we will configure the PostgreSQL database connection details required by the Streamlit dashboard.

In [30]:
# Configure PostgreSQL database details

DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "voc_analytics_db"
DB_USER = "postgres"

print("Database Configuration Prepared Successfully!")
print("----------------------------------------------")
print("Host:", DB_HOST)
print("Port:", DB_PORT)
print("Database:", DB_NAME)
print("Username:", DB_USER)

Database Configuration Prepared Successfully!
----------------------------------------------
Host: localhost
Port: 5433
Database: voc_analytics_db
Username: postgres


## 29. Create Database Connection Function

In this step, we will create a reusable function to connect the Streamlit dashboard with our PostgreSQL database.

In [31]:
# Create PostgreSQL connection function

def create_database_connection(password):

    database_url = URL.create(
        drivername="postgresql+psycopg2",
        username=DB_USER,
        password=password,
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME
    )

    engine = create_engine(database_url)

    return engine


print("Database Connection Function Created Successfully!")

Database Connection Function Created Successfully!


## 30. Create Data Fetching Function

In this step, we will create a reusable function to retrieve data from PostgreSQL database tables and views.

In [32]:
# Create database data fetching function

def fetch_data(engine, query):

    with engine.connect() as connection:
        data = pd.read_sql(text(query), connection)

    return data


print("Data Fetching Function Created Successfully!")

Data Fetching Function Created Successfully!


## 31. Create Streamlit Dashboard Structure

In this step, we will write the initial Streamlit application code into the dashboard/app.py file and configure its basic layout.

In [33]:
# Create initial Streamlit dashboard application

dashboard_code = r'''
import streamlit as st
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Page Configuration
st.set_page_config(
    page_title="Customer Intelligence Dashboard",
    page_icon="📊",
    layout="wide"
)

# Dashboard Title
st.title("AI-Powered Customer Intelligence")
st.subheader("Voice of Customer Analytics Platform")

st.markdown("""
This dashboard provides customer sentiment analysis,
product rating insights, feedback themes and business recommendations.
""")

# Database Configuration
DB_HOST = "localhost"
DB_PORT = 5433
DB_NAME = "voc_analytics_db"
DB_USER = "postgres"

# Database Password
password = st.sidebar.text_input(
    "PostgreSQL Password",
    type="password"
)

@st.cache_resource
def create_database_connection(password):

    database_url = URL.create(
        drivername="postgresql+psycopg2",
        username=DB_USER,
        password=password,
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME
    )

    return create_engine(database_url)

# Connect to Database
if password:

    try:
        engine = create_database_connection(password)

        with engine.connect() as connection:
            connection.execute(text("SELECT 1"))

        st.sidebar.success("Database Connected Successfully!")

    except Exception as error:
        st.sidebar.error("Database Connection Failed")
        st.error(f"Connection Error: {error}")

else:
    st.info("Please enter your PostgreSQL password in the sidebar to connect to the database.")
'''

dashboard_file.write_text(dashboard_code, encoding="utf-8")

print("Initial Streamlit Dashboard Created Successfully!")
print("File Location:", dashboard_file)
print("Total Characters:", len(dashboard_code))

Initial Streamlit Dashboard Created Successfully!
File Location: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py
Total Characters: 1532


## 32. Add Dashboard KPI Cards

In this step, we will display important customer analytics metrics such as total reviews, positive reviews, negative reviews and average product rating.

In [34]:
# Add KPI Cards to Streamlit Dashboard

dashboard_code = dashboard_file.read_text(encoding="utf-8")

kpi_code = '''

# Dashboard KPI Section
if password and "engine" in locals():

    try:
        with engine.connect() as connection:

            total_reviews = pd.read_sql(
                text("SELECT COUNT(*) AS total FROM customer_reviews"),
                connection
            ).iloc[0]["total"]

            sentiment_data = pd.read_sql(
                text("""
                    SELECT sentiment, COUNT(*) AS total
                    FROM customer_reviews
                    GROUP BY sentiment
                """),
                connection
            )

            average_rating = pd.read_sql(
                text("""
                    SELECT ROUND(AVG(rating)::numeric, 2) AS avg_rating
                    FROM customer_reviews
                """),
                connection
            ).iloc[0]["avg_rating"]

        sentiment_counts = dict(
            zip(sentiment_data["sentiment"], sentiment_data["total"])
        )

        positive_reviews = sentiment_counts.get("Positive", 0)
        negative_reviews = sentiment_counts.get("Negative", 0)

        st.divider()
        st.header("Customer Analytics Overview")

        col1, col2, col3, col4 = st.columns(4)

        col1.metric("Total Reviews", f"{total_reviews:,}")
        col2.metric("Positive Reviews", f"{positive_reviews:,}")
        col3.metric("Negative Reviews", f"{negative_reviews:,}")
        col4.metric("Average Rating", f"{average_rating}/5")

    except Exception as error:
        st.error(f"Unable to load dashboard metrics: {error}")
'''

dashboard_code += kpi_code

dashboard_file.write_text(dashboard_code, encoding="utf-8")

print("Dashboard KPI Cards Code Added Successfully!")
print("Updated File:", dashboard_file)

Dashboard KPI Cards Code Added Successfully!
Updated File: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py


## 33. Add Sentiment Distribution Chart

In this step, we will visualize the distribution of positive, neutral and negative customer reviews using a bar chart.

In [3]:
# Add Sentiment Distribution Chart

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence"
)

dashboard_file = PROJECT_ROOT / "dashboard" / "app.py"

dashboard_code = dashboard_file.read_text(encoding="utf-8")

sentiment_chart_code = '''

# Sentiment Distribution Section
if password:

    try:
        sentiment_chart_data = fetch_data(
            engine,
            """
            SELECT sentiment_label, COUNT(*) AS total_reviews
            FROM customer_reviews
            GROUP BY sentiment_label
            ORDER BY total_reviews DESC
            """
        )

        st.divider()
        st.header("Customer Sentiment Analysis")

        st.bar_chart(
            sentiment_chart_data.set_index("sentiment_label"),
            y="total_reviews"
        )

        st.dataframe(
            sentiment_chart_data,
            use_container_width=True,
            hide_index=True
        )

    except Exception as error:
        st.error(f"Unable to load sentiment analysis: {error}")
'''

dashboard_code += sentiment_chart_code

dashboard_file.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Sentiment Distribution Chart Code Added Successfully!")
print("Updated File:", dashboard_file)

Sentiment Distribution Chart Code Added Successfully!
Updated File: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py


## 34. Add Product Rating Distribution Chart

In this step, we will analyze the distribution of customer ratings from 1 to 5 stars using a bar chart.

In [4]:
# Add Product Rating Distribution Chart

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence"
)

dashboard_file = PROJECT_ROOT / "dashboard" / "app.py"

dashboard_code = dashboard_file.read_text(encoding="utf-8")

rating_chart_code = '''

# Product Rating Distribution Section
if password:

    try:
        rating_data = fetch_data(
            engine,
            """
            SELECT rating, COUNT(*) AS total_reviews
            FROM customer_reviews
            GROUP BY rating
            ORDER BY rating
            """
        )

        st.divider()
        st.header("Product Rating Distribution")

        st.bar_chart(
            rating_data.set_index("rating"),
            y="total_reviews"
        )

        st.dataframe(
            rating_data,
            use_container_width=True,
            hide_index=True
        )

    except Exception as error:
        st.error(f"Unable to load product rating analysis: {error}")
'''

dashboard_code += rating_chart_code

dashboard_file.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Product Rating Distribution Chart Added Successfully!")
print("Updated File:", dashboard_file)

Product Rating Distribution Chart Added Successfully!
Updated File: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py


## 35. Add Customer Feedback Theme Analysis

In this step, we will analyze the major themes identified in customer reviews, such as Price and Value, Product Quality, Product Performance, Packaging, Delivery Experience and Customer Service.

In [5]:
# Add Customer Feedback Theme Analysis

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence"
)

dashboard_file = PROJECT_ROOT / "dashboard" / "app.py"

dashboard_code = dashboard_file.read_text(encoding="utf-8")

theme_analysis_code = '''

# Customer Feedback Theme Analysis Section
if password:

    try:
        theme_data = fetch_data(
            engine,
            """
            SELECT
                TRIM(theme) AS theme,
                COUNT(*) AS total_reviews
            FROM customer_reviews,
                 LATERAL UNNEST(STRING_TO_ARRAY(matched_themes, ';')) AS theme
            WHERE matched_themes IS NOT NULL
              AND TRIM(theme) <> ''
            GROUP BY TRIM(theme)
            ORDER BY total_reviews DESC
            """
        )

        st.divider()
        st.header("Customer Feedback Theme Analysis")

        st.bar_chart(
            theme_data.set_index("theme"),
            y="total_reviews"
        )

        st.dataframe(
            theme_data,
            use_container_width=True,
            hide_index=True
        )

    except Exception as error:
        st.error(f"Unable to load theme analysis: {error}")
'''

dashboard_code += theme_analysis_code

dashboard_file.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Customer Feedback Theme Analysis Added Successfully!")
print("Updated File:", dashboard_file)

Customer Feedback Theme Analysis Added Successfully!
Updated File: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py


## 36. Add Negative Feedback Analysis

In this step, we will identify the major themes associated with negative customer reviews and analyze areas that require business improvement.

In [6]:
# Add Negative Feedback Analysis

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence"
)

dashboard_file = PROJECT_ROOT / "dashboard" / "app.py"

dashboard_code = dashboard_file.read_text(encoding="utf-8")

negative_analysis_code = '''

# Negative Feedback Analysis Section
if password:

    try:
        negative_data = fetch_data(
            engine,
            """
            SELECT
                TRIM(theme) AS theme,
                COUNT(*) AS negative_reviews
            FROM customer_reviews,
                 LATERAL UNNEST(STRING_TO_ARRAY(matched_themes, ';')) AS theme
            WHERE sentiment_label = 'Negative'
              AND matched_themes IS NOT NULL
              AND TRIM(theme) <> ''
            GROUP BY TRIM(theme)
            ORDER BY negative_reviews DESC
            """
        )

        st.divider()
        st.header("Negative Customer Feedback Analysis")

        st.bar_chart(
            negative_data.set_index("theme"),
            y="negative_reviews"
        )

        st.dataframe(
            negative_data,
            use_container_width=True,
            hide_index=True
        )

    except Exception as error:
        st.error(f"Unable to load negative feedback analysis: {error}")
'''

dashboard_code += negative_analysis_code

dashboard_file.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Negative Feedback Analysis Added Successfully!")
print("Updated File:", dashboard_file)

Negative Feedback Analysis Added Successfully!
Updated File: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py


## 37. Add Business Recommendations

In this step, we will display business recommendations based on customer feedback analysis to help businesses improve customer satisfaction and product experience.

In [7]:
# Add Business Recommendations Section

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence"
)

dashboard_file = PROJECT_ROOT / "dashboard" / "app.py"

dashboard_code = dashboard_file.read_text(encoding="utf-8")

recommendations_code = '''

# Business Recommendations Section
if password:

    try:
        recommendations_data = fetch_data(
            engine,
            """
            SELECT *
            FROM business_recommendations
            """
        )

        st.divider()
        st.header("Business Recommendations")

        st.dataframe(
            recommendations_data,
            use_container_width=True,
            hide_index=True
        )

    except Exception as error:
        st.error(f"Unable to load business recommendations: {error}")

# Dashboard Footer
st.divider()
st.caption(
    "AI-Powered Customer Intelligence and Voice of Customer Analytics Platform"
)
'''

dashboard_code += recommendations_code

dashboard_file.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Business Recommendations Section Added Successfully!")
print("Updated File:", dashboard_file)

Business Recommendations Section Added Successfully!
Updated File: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\dashboard\app.py


## 38. Verify Dashboard Database Columns

In this step, we will verify the column names of the customer_reviews table before running the Streamlit dashboard.

In [8]:
# Verify customer_reviews table columns

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass
import pandas as pd

db_password = getpass("Enter PostgreSQL Password: ")

schema_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=db_password,
    host="localhost",
    port=5433,
    database="voc_analytics_db"
)

schema_engine = create_engine(schema_url)

with schema_engine.connect() as connection:
    column_data = pd.read_sql(
        text("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_name = 'customer_reviews'
            ORDER BY ordinal_position
        """),
        connection
    )

display(column_data)

print("Database Column Verification Completed!")

Enter PostgreSQL Password:  ········


,column_name,data_type
0,review_id,bigint
1,product_id,text
2,date_added,text
3,date_updated,text
4,product_name,text
5,asins,text
6,brand,text
7,categories,text
8,primary_categories,text
9,image_urls,text


Database Column Verification Completed!


## 39. Finalize Streamlit Dashboard Application

In this step, we will integrate all dashboard components into a single Streamlit application, including KPI cards, sentiment analysis, product ratings, feedback themes, negative feedback and business recommendations.

In [9]:
# Finalize Streamlit Dashboard Application

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence"
)

dashboard_file = PROJECT_ROOT / "dashboard" / "app.py"

dashboard_code = r'''
import streamlit as st
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


# --------------------------------------------------
# 1. PAGE CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="Customer Intelligence Dashboard",
    page_icon="📊",
    layout="wide"
)

st.title("AI-Powered Customer Intelligence")
st.subheader("Voice of Customer Analytics Platform")

st.markdown(
    "An interactive dashboard for customer sentiment, "
    "product ratings, feedback themes and business insights."
)


# --------------------------------------------------
# 2. DATABASE CONFIGURATION
# --------------------------------------------------

DB_HOST = "localhost"
DB_PORT = 5433
DB_NAME = "voc_analytics_db"
DB_USER = "postgres"


@st.cache_resource
def create_database_connection(password):

    database_url = URL.create(
        drivername="postgresql+psycopg2",
        username=DB_USER,
        password=password,
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME
    )

    return create_engine(database_url)


def fetch_data(engine, query):

    with engine.connect() as connection:
        return pd.read_sql(text(query), connection)


# --------------------------------------------------
# 3. DATABASE LOGIN
# --------------------------------------------------

st.sidebar.header("Database Connection")

password = st.sidebar.text_input(
    "PostgreSQL Password",
    type="password"
)

if password:

    try:
        engine = create_database_connection(password)

        with engine.connect() as connection:
            connection.execute(text("SELECT 1"))

        st.sidebar.success("Database Connected Successfully!")

        # --------------------------------------------------
        # 4. KPI CARDS
        # --------------------------------------------------

        kpi_data = fetch_data(
            engine,
            """
            SELECT
                COUNT(*) AS total_reviews,
                ROUND(AVG(rating)::numeric, 2) AS average_rating,
                COUNT(*) FILTER (
                    WHERE sentiment_label = 'Positive'
                ) AS positive_reviews,
                COUNT(*) FILTER (
                    WHERE sentiment_label = 'Negative'
                ) AS negative_reviews
            FROM customer_reviews
            """
        )

        total_reviews = int(kpi_data.iloc[0]["total_reviews"])
        average_rating = kpi_data.iloc[0]["average_rating"]
        positive_reviews = int(kpi_data.iloc[0]["positive_reviews"])
        negative_reviews = int(kpi_data.iloc[0]["negative_reviews"])

        st.divider()
        st.header("Customer Analytics Overview")

        col1, col2, col3, col4 = st.columns(4)

        col1.metric("Total Reviews", f"{total_reviews:,}")
        col2.metric("Positive Reviews", f"{positive_reviews:,}")
        col3.metric("Negative Reviews", f"{negative_reviews:,}")
        col4.metric("Average Rating", f"{average_rating}/5")

        # --------------------------------------------------
        # 5. SENTIMENT DISTRIBUTION
        # --------------------------------------------------

        sentiment_data = fetch_data(
            engine,
            """
            SELECT
                sentiment_label,
                COUNT(*) AS total_reviews
            FROM customer_reviews
            GROUP BY sentiment_label
            ORDER BY total_reviews DESC
            """
        )

        st.divider()
        st.header("Customer Sentiment Analysis")

        st.bar_chart(
            sentiment_data.set_index("sentiment_label"),
            y="total_reviews"
        )

        st.dataframe(
            sentiment_data,
            use_container_width=True,
            hide_index=True
        )

        # --------------------------------------------------
        # 6. PRODUCT RATING DISTRIBUTION
        # --------------------------------------------------

        rating_data = fetch_data(
            engine,
            """
            SELECT
                rating,
                COUNT(*) AS total_reviews
            FROM customer_reviews
            GROUP BY rating
            ORDER BY rating
            """
        )

        st.divider()
        st.header("Product Rating Distribution")

        st.bar_chart(
            rating_data.set_index("rating"),
            y="total_reviews"
        )

        # --------------------------------------------------
        # 7. CUSTOMER FEEDBACK THEMES
        # --------------------------------------------------

        theme_data = fetch_data(
            engine,
            """
            SELECT
                TRIM(t.theme) AS theme,
                COUNT(*) AS total_reviews
            FROM customer_reviews AS cr
            CROSS JOIN LATERAL
                UNNEST(STRING_TO_ARRAY(cr.matched_themes, ';'))
                AS t(theme)
            WHERE cr.matched_themes IS NOT NULL
              AND TRIM(t.theme) <> ''
            GROUP BY TRIM(t.theme)
            ORDER BY total_reviews DESC
            """
        )

        st.divider()
        st.header("Customer Feedback Theme Analysis")

        st.bar_chart(
            theme_data.set_index("theme"),
            y="total_reviews"
        )

        st.dataframe(
            theme_data,
            use_container_width=True,
            hide_index=True
        )

        # --------------------------------------------------
        # 8. NEGATIVE FEEDBACK ANALYSIS
        # --------------------------------------------------

        negative_data = fetch_data(
            engine,
            """
            SELECT
                TRIM(t.theme) AS theme,
                COUNT(*) AS negative_reviews
            FROM customer_reviews AS cr
            CROSS JOIN LATERAL
                UNNEST(STRING_TO_ARRAY(cr.matched_themes, ';'))
                AS t(theme)
            WHERE cr.sentiment_label = 'Negative'
              AND cr.matched_themes IS NOT NULL
              AND TRIM(t.theme) <> ''
            GROUP BY TRIM(t.theme)
            ORDER BY negative_reviews DESC
            """
        )

        st.divider()
        st.header("Negative Customer Feedback Analysis")

        st.bar_chart(
            negative_data.set_index("theme"),
            y="negative_reviews"
        )

        st.dataframe(
            negative_data,
            use_container_width=True,
            hide_index=True
        )

        # --------------------------------------------------
        # 9. BUSINESS RECOMMENDATIONS
        # --------------------------------------------------

        recommendations_data = fetch_data(
            engine,
            """
            SELECT area, finding, recommendation
            FROM business_recommendations
            ORDER BY recommendation_id
            """
        )

        st.divider()
        st.header("Business Recommendations")

        st.dataframe(
            recommendations_data,
            use_container_width=True,
            hide_index=True
        )

        st.success("Dashboard Data Loaded Successfully!")

    except Exception as error:
        st.sidebar.error("Database Connection or Query Failed")
        st.error(f"Error Details: {error}")

else:
    st.info(
        "Please enter your PostgreSQL password in the sidebar "
        "to load the dashboard."
    )


# --------------------------------------------------
# 10. DASHBOARD FOOTER
# --------------------------------------------------

st.divider()

st.caption(
    "AI-Powered Customer Intelligence and Voice of Customer Analytics Platform"
)
'''

dashboard_file.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Complete Streamlit Dashboard Code Saved Successfully!")
print("-----------------------------------------------")
print("File:", dashboard_file.name)
print("Total Characters:", len(dashboard_code))
print("File Exists:", dashboard_file.exists())

Complete Streamlit Dashboard Code Saved Successfully!
-----------------------------------------------
File: app.py
Total Characters: 7637
File Exists: True
